# History matches and category depth

**Question:** which kinds of past behaviour provide usable evidence for ranking products in the current search?

Use the same uniform sample of **5,000 complete search records**, seed **4222**, as the query diagnostics. Compare exact-product, brand, shop, and category L1–L4 matches. Separately use all history, history with a recorded query, and query-less history (`-1`). Query presence does not establish that a past search resembles the current search.

Scan the full catalog and retain metadata for both candidates and historical products. Unknown metadata cannot match other unknown metadata. Exact-product identity is available even without catalog metadata. Category matches use the complete ancestor path (e.g. L1/L2/L3 for L3), protecting against a child ID reused under different parents. Hierarchy/missing-ID checks cover the full catalog; history/ranking checks cover the sampled records.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/BT4222 Group Project/data")
    REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/history_match_diagnostics.py").is_file()), None)
except ImportError:
    REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/history_match_diagnostics.py").is_file())
    DATA_DIR = REPO_ROOT / "data"
if REPO_ROOT is None:
    raise FileNotFoundError("Open the repository checkout containing scripts/history_match_diagnostics.py first.")
sys.path.insert(0, str(REPO_ROOT))
from scripts.history_match_diagnostics import run_history_validation, validate_match_helpers
OUTPUT_DIR = REPO_ROOT / "outputs/interim/tables/history_validation"
QUERY_SAMPLE = REPO_ROOT / "outputs/interim/tables/query_validation/query_diagnostics_per_search.csv"
SAMPLE_SIZE, SEED = 5000, 4222
validate_match_helpers()
print("Missing-value, repeated-action, source alignment and category-path checks passed.")


Missing-value, repeated-action, source alignment and category-path checks passed.


## Definitions and validation

For each candidate, count prior interactions with the same product, brand, shop or category path. Do not assign action-strength weights here. Count scores can vary even when every candidate has at least one match, so we measure both count variation and binary match variation.

- **Coverage:** share of eligible candidate occurrences with at least one observed match. Candidates with unavailable metadata are excluded from that feature's coverage/rate comparison.
- **Attribute variation:** whether known candidates in a search have different brands/shops/categories.
- **Count variation:** whether their history-match counts differ, allowing a count score to reorder them.
- **Engagement rates:** label > 0 with versus without a match. Pooled ratios can reflect different searches/history lengths. A second comparison computes the difference within each eligible search and gives equal weight to searches, with 2,000 bootstrap resamples.
- **Ranking:** count score on complete lists; NDCG@10, linear gains 0/1/2/3. Ties use expected random order, never file position. Exclude all-zero lists. This is an exploratory heuristic comparison, not held-out model evaluation.

Synthetic cases guard against matching missing metadata, reused child IDs, missing L4 and incorrectly aligned query sources. Counts are additionally audited against direct comparisons on actual records. Source-specific counts must sum to all-history counts; deeper category-path counts cannot exceed ancestor counts.


In [2]:
summary, matches, ranking, depth, metadata_coverage, per_search = run_history_validation(
    DATA_DIR, OUTPUT_DIR, sample_size=SAMPLE_SIZE, seed=SEED,
    expected_sample_path=QUERY_SAMPLE if QUERY_SAMPLE.is_file() else None
)


Loaded 5,000 complete records; need metadata for 729,008 products
Scanned 12,141,247 catalog rows; retained 698,777 matching rows
{
  "seed": 4222,
  "total_records": 173831,
  "sample_records": 5000,
  "candidate_occurrences": 449052,
  "history_events": 758712,
  "search_history_events": 267442,
  "queryless_history_events": 491270,
  "catalog_rows_scanned": 12141247,
  "distinct_behavior_products_requested": 729008,
  "distinct_behavior_products_with_metadata": 698777,
  "catalog_missing_id_counts": {
    "brand": 0,
    "shop": 103162,
    "L1": 0,
    "L2": 0,
    "L3": 0,
    "L4": 0
  },
  "catalog_L4_equals_L3_rows": 0,
  "category_child_ids_with_multiple_parents": {
    "L2": 0,
    "L3": 17,
    "L4": 33
  },
  "candidate_metadata_usable_share": {
    "product": 1.0,
    "brand": 0.9956352493697834,
    "shop": 0.9867209142816422,
    "L1": 0.9956352493697834,
    "L2": 0.9956352493697834,
    "L3": 0.9956352493697834,
    "L4": 0.9956352493697834
  },
  "independent_actual_r

In [3]:
columns = ["level", "matched_share", "engagement_with_match", "engagement_without_match",
           "attribute_varies_search_share", "count_varies_search_share", "contrast_eligible_searches"]
display(matches.loc[matches.source == "all", columns])
display(metadata_coverage)
display(ranking.pivot(index="level", columns="source", values="ndcg_at_10").reindex(["product", "brand", "shop", "L1", "L2", "L3", "L4"]))
display(depth)


,level,matched_share,engagement_with_match,engagement_without_match,attribute_varies_search_share,count_varies_search_share,contrast_eligible_searches
0,product,0.007899,0.258246,0.018875,0.9976,0.2952,1476
1,brand,0.226933,0.029923,0.018118,0.8994,0.5944,2950
2,shop,0.071552,0.064598,0.017497,0.9716,0.5078,2533
3,L1,0.751666,0.021627,0.018284,0.6634,0.5896,2120
4,L2,0.574879,0.022745,0.018162,0.7890,0.6070,2740
5,L3,0.406552,0.024774,0.018072,0.8904,0.5596,2701
6,L4,0.375891,0.025557,0.017930,0.9116,0.5592,2714


history_events  ...  metadata_usable_share
source    level                    ...                       
all       L1               758712  ...               0.951851
          L2               758712  ...               0.951851
          L3               758712  ...               0.951851
          L4               758712  ...               0.951851
          brand            758712  ...               0.951851
          product          758712  ...               1.000000
          shop             758712  ...               0.948302
queryless L1               491270  ...               0.952307
          L2               491270  ...               0.952307
          L3               491270  ...               0.952307
          L4               491270  ...               0.952307
          brand            491270  ...               0.952307
          product          491270  ...               1.000000
          shop             491270  ...               0.949193
search    L1               267442  ...               0.951014
          L2               267442  ...               0.951014
          L3               267442  ...               0.951014
          L4               267442  ...               0.951014
          brand            267442  ...               0.951014
          product          267442  ...               1.000000
          shop             267442  ...               0.946665

[21 rows x 3 columns]

source,all,queryless,search
level,,,
product,0.180903,0.149513,0.139776
brand,0.119796,0.111958,0.109040
shop,0.170576,0.149607,0.145406
L1,0.084910,0.084261,0.083908
L2,0.088209,0.087624,0.086609
L3,0.094472,0.092961,0.090668
L4,0.095776,0.094253,0.091753


,source,comparison,ndcg_difference,ci_low,ci_high
0,all,L2 minus L1,0.003299,0.001868,0.004689
1,all,L3 minus L2,0.006263,0.004511,0.008009
2,all,L4 minus L3,0.001303,0.000401,0.002190
3,search,L2 minus L1,0.002700,0.001398,0.004046
4,search,L3 minus L2,0.004059,0.002482,0.005722
5,search,L4 minus L3,0.001085,0.000324,0.001844
6,queryless,L2 minus L1,0.003363,0.002084,0.004687
7,queryless,L3 minus L2,0.005336,0.003691,0.006930
8,queryless,L4 minus L3,0.001293,0.000411,0.002111


## Interpretation limits

A broader category match covers more products but may provide less discrimination; a deeper match can be more specific but rarer. The measurements decide whether that trade-off is present here. Category variation alone does not prove that it predicts engagement.

Comparing search-associated and query-less history measures two available feature sources. Different amounts and action-type mixes of history can explain performance differences; this is not evidence that one source is inherently more reliable. Nor does this analysis test current-query-conditioned weighting or a learned combination of features.

Treat non-interaction as exposed-but-not-engaged, not dislike. Missing historical metadata can hide genuine matches. Catalog checks describe observed IDs; brand, shop and categories are separate kinds of matches, not one strict specificity hierarchy.


## Findings for this run (5,000 records, seed 4222)

- Exact-product matches cover only 0.79% of candidates but have high engagement association. Shop matches cover 7.16%, brand 22.69%; both produce useful simple ranking scores.
- Category history matches cover 75.17%, 57.49%, 40.66%, 37.59% at L1–L4. Candidates have differing known category paths in 66.34%, 78.90%, 89.04%, 91.16% of lists. The coarse categories are not generally constant.
- Simple category-count NDCG@10 increases from 0.0849 (L1) to 0.0882 (L2), 0.0945 (L3), 0.0958 (L4). L4 adds a small gain over L3; this does not prove its incremental value in a learned model.
- Query-less actions are 64.75% of sampled history and provide useful signals. Keep them available; the source comparison is not controlled for history volume or action mix.
- Metadata covers 99.56% of candidate occurrences and 95.19% of historical actions. Full-catalog checks found no missing L4 IDs among present products, but 17 L3 and 33 L4 IDs appear with multiple parents. Use ancestor paths for category matching.

All counts and ranking checks passed. See `docs/FINDINGS.md` for denominators, intervals and interpretation. These are exploratory heuristics, not held-out model results. Changing the sample configuration requires updating this dated interpretation.
